### Use streamable http protocol for connecting to MCP server

In this demo, we will build an Agent using OpenAI SDK Agent framework.
We will leverage  streamable http protocol to connect Agent to MCP server.

We will use Uniprot MCP server for this. We will run this MCP server locally which will expose the server in localhost and then we will connect Agent to this MCP server via http protocol.

Steps
1. Download uniprot MCP server github
2. Install MCP server
3. Start the server
4. Build Agent with MCP connection
5. Interact with Agent with a chat interface



#### Uniprot MCP server
This is not official MCP server from uniprot

https://github.com/QuentinCody/uniprot-mcp-server


In [ ]:
# Clone the repo by 
# git clone https://github.com/QuentinCody/uniprot-mcp-server.git
# cd  uniprot-mcp-server

# install the server
# npm install

# Launch server. Opens at http://localhost:8787
# npm run dev

#### Import libraries

We will mainly import MCPServerStreamableHttp

In [3]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, SQLiteSession
from agents.mcp import MCPServerStreamableHttp

#### Load Environment

In [2]:
load_dotenv(override=True)

True

#### Lets connect to Uniprot MCP server
Check available tools in the MCP server

The server is running at http://localhost:8787

The url for mcp would be http://localhost:8787/mcp

![alt text](image.png)

In [9]:
# https://github.com/QuentinCody/uniprot-mcp-server
async with MCPServerStreamableHttp(params = {
            "url": "http://localhost:8787/mcp",
            "timeout": 60,
            "sse_read_timeout": 300,
        }, 
        max_retry_attempts=2,
        retry_backoff_seconds_base=2.0,
        client_session_timeout_seconds=60,
        cache_tools_list= True
        ) as server:
    tool_lists = await server.list_tools()

tool_lists

[Tool(name='uniprot_search', title=None, description='Search UniProtKB database with comprehensive filtering and pagination. Supports small to large result sets with automatic staging.', inputSchema={'type': 'object', 'properties': {'query': {'type': 'string', 'description': "UniProtKB search query. Examples: 'organism_id:9606 AND reviewed:true', 'gene:BRCA1', 'length:[100 TO 500]'"}, 'format': {'type': 'string', 'enum': ['json', 'tsv', 'fasta', 'xml'], 'default': 'json', 'description': 'Response format'}, 'fields': {'type': 'string', 'description': 'Comma-separated fields to return (only for json/tsv formats)'}, 'size': {'type': 'number', 'default': 25, 'description': 'Results per page (max 500)'}, 'sort': {'type': 'string', 'description': "Sort field and direction (e.g., 'score desc', 'length asc')"}, 'facets': {'type': 'string', 'description': 'Comma-separated facet fields for aggregation'}, 'compressed': {'type': 'boolean', 'default': False, 'description': 'Enable response compress

#### Build Agent with Uniprot MCP server

In [ ]:
model = "gpt-4.1-nano"

agent_instructions = (
        "You are a expert in Uniprot"
    )

session = SQLiteSession("uniprot_session")

async def chat(message, history):
    async with MCPServerStreamableHttp(params = {
                "url": "http://localhost:8787/mcp",
                "timeout": 60,
                "sse_read_timeout": 300,
            }, 
            max_retry_attempts=2,
            retry_backoff_seconds_base=2.0,
            client_session_timeout_seconds=60,
            cache_tools_list= True
            ) as server:
        uniprot_agent = Agent(name="uniprot_agent",
                        instructions=agent_instructions,
                        model=model,
                        mcp_servers=[server])
        result = await Runner.run(uniprot_agent, message)
        return result.final_output

# Some example to try
# get uniprot ids for gene TP53. just 5 is enough. only uniprot ids
# Next ask describe in 2 lines about P04637

import gradio as gr
gr.ChatInterface(
    chat,
    title="Uniprot Chatbot",
    description="Chat with the Uniprot Expert"
).launch() 

/mnt/c/Users/PriyabrataPanigrahi/Downloads/ai/aienv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
